In [ ]:
import glob

data_dir = "/nfs/tcdynasty/tang/tcradar/"

nc_files = sorted(glob.glob(f"{data_dir}/*.nc"))

print("Available files:")
for f in nc_files:
    print(f)


In [ ]:
import xarray as xr

base_dir = "/nfs/tcdynasty/tang/tcradar/"

merge_file = base_dir + "tc_radar_v3m_1997_2019_xy_rel_merge_ships.nc"
swath_file = base_dir + "tc_radar_v3m_1997_2019_xy_rel_swath_ships.nc"

ds_merge = xr.open_dataset(merge_file)
ds_swath = xr.open_dataset(swath_file)

print("MERGE dataset:")
print(ds_merge)

print("\nSWATH dataset:")
print(ds_swath)


In [ ]:
[var for var in ds_merge.data_vars if "storm" in var.lower()
 or "name" in var.lower()
 or "year" in var.lower()
 or "basin" in var.lower()]


In [ ]:
[var for var in ds_merge.coords]


In [ ]:
ds_merge["storm_name"].isel(num_cases=slice(0,10))


In [ ]:
import numpy as np

storm_names = np.char.upper(ds_merge["storm_name"].values.astype(str))
years = ds_merge["merge_year"].values

eduard_idx = np.where(
    (storm_names == "EDOUARD") & (years == 2014)
)[0]

eduard_idx


In [ ]:
ds_edouard = ds_merge.isel(num_cases=[114, 115, 116, 117, 118, 119])

print(ds_edouard)


In [ ]:
import numpy as np

print("Storm:", np.unique(ds_edouard["storm_name"].values))
print("Year:", np.unique(ds_edouard["merge_year"].values))
print("Number of cases:", ds_edouard.dims["num_cases"])


In [ ]:
#----------------          LOOKING AT TILT VARIABLES          --------------

In [ ]:
case = 350

refl_3d = ds_swath['recentered_reflectivity'].values[case]

print(refl_3d.shape)

In [ ]:
# Re-create coordinates from ds_swath
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

print("East shape:", east.shape)
print("North shape:", north.shape)
print("Height shape:", height.shape)

In [ ]:
centers_x = []
centers_y = []

X, Y = np.meshgrid(east, north)

for k in range(len(height)):

    slice2d = refl_3d[:, :, k]

    # Focus on inner-core reflectivity
    mask = slice2d >= 20  # threshold in dBZ

    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    weights = slice2d * mask

    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)

In [ ]:
tilt_mag = np.sqrt(centers_x**2 + centers_y**2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

plt.figure(figsize=(5,6))
plt.plot(tilt_mag, height, linewidth=2)
plt.xlabel("Horizontal Displacement (km)")
plt.ylabel("Height (km)")
plt.title("Vortex Tilt Profile")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(6,6))

# Plan view at 1 km
height_idx = np.argmin(np.abs(height - 1.0))

plt.imshow(
    refl_3d[:, :, height_idx],
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=-10, vmax=50
)

# Select key levels (every ~3 km)
levels_to_plot = [0, 3, 6, 9, 12]
idx_levels = [np.argmin(np.abs(height - h)) for h in levels_to_plot]

for i in idx_levels:
    plt.scatter(
        centers_x[i],
        centers_y[i],
        s=120,
        label=f"{height[i]:.1f} km"
    )

# Connect surface to top (bulk tilt vector)
top_idx = np.nanargmax(height)

plt.plot(
    [centers_x[0], centers_x[top_idx]],
    [centers_y[0], centers_y[top_idx]],
    'k--',
    linewidth=2,
    label="Bulk Tilt Vector"
)

plt.scatter(0, 0, color='red', s=140, label="Surface Center")

# -----------------------------
# 🔎 Zoom into inner 150 km radius
# -----------------------------
zoom_radius = 150
plt.xlim(-zoom_radius, zoom_radius)
plt.ylim(-zoom_radius, zoom_radius)

plt.xlabel("Eastward Distance (km)")
plt.ylabel("Northward Distance (km)")
plt.title("Vortex Tilt Structure (Zoomed In)")
plt.legend()
plt.colorbar(label="dBZ")
plt.show()

In [ ]:
# ---------------------         COMBINING THE PLOT ----------------------

In [ ]:
# ====================================
# USER INPUT
# ====================================
case = 350       # <-- change this to any case number you want
zoom_radius = 100  # km for plan view zoom

# ====================================
# IMPORTS
# ====================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ====================================
# EXTRACT DATA FOR THE SELECTED CASE
# ====================================
refl_3d = ds_swath['recentered_reflectivity'].values[case]
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# Create meshgrid for centroid calculation
X, Y = np.meshgrid(east, north)

# ------------------------------------
# COMPUTE REFLECTIVITY CENTROIDS
# ------------------------------------
centers_x = []
centers_y = []

for k in range(len(height)):
    slice2d = refl_3d[:, :, k]
    mask = slice2d >= 20  # threshold in dBZ
    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue
    weights = slice2d * mask
    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)
    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)

# ------------------------------------
# COMPUTE TILT MAGNITUDE
# ------------------------------------
tilt_mag = np.sqrt(centers_x**2 + centers_y**2)

# ====================================
# PLOT BOTH IMAGES SIDE BY SIDE
# ====================================
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# ---- LEFT: Tilt vs Height ----
axes[0].plot(tilt_mag, height, linewidth=2, color='blue')
axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile")
axes[0].grid(True)

# ---- RIGHT: Plan View Tilt Structure ----
height_idx = np.argmin(np.abs(height - 1.0))

im = axes[1].imshow(
    refl_3d[:, :, height_idx],
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=-10, vmax=50
)

# Plot key height levels (~every 3 km)
levels_to_plot = [0, 3, 6, 9, 12]
idx_levels = [np.argmin(np.abs(height - h)) for h in levels_to_plot]

for i in idx_levels:
    axes[1].scatter(
        centers_x[i],
        centers_y[i],
        s=120,
        label=f"{height[i]:.1f} km"
    )

# Bulk tilt vector
top_idx = np.nanargmax(height)
axes[1].plot(
    [centers_x[0], centers_x[top_idx]],
    [centers_y[0], centers_y[top_idx]],
    'k--',
    linewidth=2,
    label="Bulk Tilt Vector"
)

axes[1].scatter(0, 0, color='red', s=140, label="Surface Center")

# Zoom in
axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_xlabel("Eastward Distance (km)")
axes[1].set_ylabel("Northward Distance (km)")
axes[1].set_title("Vortex Tilt Structure (Zoomed In)")
axes[1].legend()
fig.colorbar(im, ax=axes[1], label="dBZ")

plt.tight_layout()
plt.show()

In [ ]:
#displacement relative to the surface centroid:

In [ ]:
# ====================================
# USER INPUT
# ====================================
case = 350       # <-- change this to any case number you want
zoom_radius = 100  # km for plan view zoom

# ====================================
# IMPORTS
# ====================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ====================================
# EXTRACT DATA FOR THE SELECTED CASE
# ====================================
refl_3d = ds_swath['recentered_reflectivity'].values[case]
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# Create meshgrid for centroid calculation
X, Y = np.meshgrid(east, north)

# ====================================
# COMPUTE REFLECTIVITY CENTROIDS
# ====================================
centers_x = []
centers_y = []

for k in range(len(height)):
    slice2d = refl_3d[:, :, k]
    mask = slice2d >= 20  # threshold in dBZ
    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue
    weights = slice2d * mask
    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)
    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)

# ====================================
# COMPUTE TILT MAGNITUDE RELATIVE TO SURFACE
# ====================================
# Find first valid (non-NaN) surface centroid
valid_idx = np.where(~np.isnan(centers_x) & ~np.isnan(centers_y))[0][0]

x_rel = centers_x - centers_x[valid_idx]
y_rel = centers_y - centers_y[valid_idx]
tilt_mag_rel = np.sqrt(x_rel**2 + y_rel**2)

# ====================================
# PLOT BOTH IMAGES SIDE BY SIDE
# ====================================
fig, axes = plt.subplots(1, 2, figsize=(14,6))

# ---- LEFT: Tilt vs Height (Relative to Surface) ----
axes[0].plot(tilt_mag_rel, height, linewidth=2, color='blue')
axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile (Relative to Surface)")
axes[0].grid(True)

# ---- RIGHT: Plan View Tilt Structure ----
height_idx = np.argmin(np.abs(height - 1.0))

im = axes[1].imshow(
    refl_3d[:, :, height_idx],
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=-10, vmax=50
)

# Plot key height levels (~every 3 km)
levels_to_plot = [0, 3, 6, 9, 12]
idx_levels = [np.argmin(np.abs(height - h)) for h in levels_to_plot]

for i in idx_levels:
    axes[1].scatter(
        centers_x[i],
        centers_y[i],
        s=120,
        label=f"{height[i]:.1f} km"
    )

# Bulk tilt vector
top_idx = np.nanargmax(height)
axes[1].plot(
    [centers_x[0], centers_x[top_idx]],
    [centers_y[0], centers_y[top_idx]],
    'k--',
    linewidth=2,
    label="Bulk Tilt Vector"
)

axes[1].scatter(0, 0, color='red', s=140, label="Surface Center")

# Zoom in
axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_xlabel("Eastward Distance (km)")
axes[1].set_ylabel("Northward Distance (km)")
axes[1].set_title("Vortex Tilt Structure (Zoomed In)")
axes[1].legend()
fig.colorbar(im, ax=axes[1], label="dBZ")

plt.tight_layout()
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 357            # Case number to analyze
zoom_radius = 100     # Radius (km) to zoom plan-view plot

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ======================================================
# EXTRACT RADAR DATA FOR SELECTED CASE
# ======================================================

# 3D reflectivity field
# dimensions: (north, east, height)
refl_3d = ds_swath['recentered_reflectivity'].values[case]

# Storm-centered coordinate grid (km)
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values

# Vertical levels (km)
height = ds_swath['height'].values


# ======================================================
# GET UTC TIME FROM SHIPS METADATA
# ======================================================

year  = int(ds_swath['year_ships'].values[case, 0])
month = int(ds_swath['month_ships'].values[case, 0])
day   = int(ds_swath['day_ships'].values[case, 0])
hour  = int(ds_swath['hour_ships'].values[case, 0])

storm_time = pd.Timestamp(year, month, day, hour)
utc_string = storm_time.strftime("%Y-%m-%d %HZ SHIPS Analysis Time")


# ======================================================
# CREATE GRID FOR CENTROID CALCULATION
# ======================================================

# Meshgrid gives the x,y location of each reflectivity pixel
X, Y = np.meshgrid(east, north)


# ======================================================
# COMPUTE REFLECTIVITY CENTROIDS BY HEIGHT
# ======================================================

centers_x = []
centers_y = []

for k in range(len(height)):

    # Reflectivity slice at this height
    refl_slice = refl_3d[:, :, k]

    # Only consider convective pixels ≥ 20 dBZ
    mask = refl_slice >= 20

    # If too few pixels, skip level
    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    # Weight centroid by reflectivity magnitude
    weights = refl_slice * mask

    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ======================================================
# COMPUTE TILT RELATIVE TO SURFACE CENTER
# ======================================================

# First valid centroid = surface reference
valid_idx = np.where(~np.isnan(centers_x) & ~np.isnan(centers_y))[0][0]

x_surface = centers_x[valid_idx]
y_surface = centers_y[valid_idx]

# Horizontal displacement from surface center
x_rel = centers_x - x_surface
y_rel = centers_y - y_surface

# Total tilt magnitude
tilt_mag = np.sqrt(x_rel**2 + y_rel**2)


# ======================================================
# CREATE FIGURE
# ======================================================

fig, axes = plt.subplots(1, 2, figsize=(14,6))

# Figure title with case info
fig.suptitle(
    f"Vortex Tilt Analysis | Case {case} | {utc_string}",
    fontsize=16,
    fontweight="bold"
)


# ======================================================
# LEFT PANEL: TILT PROFILE
# ======================================================

axes[0].plot(tilt_mag, height, linewidth=2)

axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile (Relative to Surface)")
axes[0].grid(True)


# ======================================================
# RIGHT PANEL: PLAN VIEW STRUCTURE
# ======================================================

# Height level closest to 1 km
height_idx = np.argmin(np.abs(height - 1.0))

# Reflectivity plan view
im = axes[1].imshow(
    refl_3d[:, :, height_idx],
    origin="lower",
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap="turbo",
    vmin=-10,
    vmax=50
)

# Plot centroids at selected heights
levels_to_plot = [0, 3, 6, 9, 12]
idx_levels = [np.argmin(np.abs(height - h)) for h in levels_to_plot]

for i in idx_levels:
    axes[1].scatter(
        centers_x[i],
        centers_y[i],
        s=120,
        label=f"{height[i]:.1f} km"
    )

# Bulk tilt vector (surface → highest level)
top_idx = np.nanargmax(height)

axes[1].plot(
    [centers_x[0], centers_x[top_idx]],
    [centers_y[0], centers_y[top_idx]],
    "k--",
    linewidth=2,
    label="Bulk Tilt Vector"
)

# Mark surface center
axes[1].scatter(0, 0, color="red", s=140, label="Surface Center")

# Zoom view
axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_xlabel("Eastward Distance (km)")
axes[1].set_ylabel("Northward Distance (km)")
axes[1].set_title("Vortex Tilt Structure (Plan View)")
axes[1].legend()

# Colorbar
fig.colorbar(im, ax=axes[1], label="Reflectivity (dBZ)")


# ======================================================
# FINAL LAYOUT
# ======================================================

plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 357
zoom_radius = 100   # km


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


# ======================================================
# LOAD RADAR DATA
# ======================================================

refl_3d = ds_swath['recentered_reflectivity'].values[case]

east  = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values


# ======================================================
# GET UTC TIME FROM SHIPS
# ======================================================

year  = int(ds_swath['year_ships'].values[case,0])
month = int(ds_swath['month_ships'].values[case,0])
day   = int(ds_swath['day_ships'].values[case,0])
hour  = int(ds_swath['hour_ships'].values[case,0])

storm_time = pd.Timestamp(year,month,day,hour)
utc_string = storm_time.strftime("%Y-%m-%d %HZ SHIPS Analysis Time")


# ======================================================
# CREATE GRID
# ======================================================

X, Y = np.meshgrid(east, north)


# ======================================================
# COMPUTE CENTROIDS BY HEIGHT
# ======================================================

centers_x = []
centers_y = []

for k in range(len(height)):

    refl_slice = refl_3d[:,:,k]

    mask = refl_slice >= 20

    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    weights = refl_slice * mask

    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ======================================================
# DEFINE 2-KM VORTEX CENTER
# ======================================================

center_level = 2.0
center_idx = np.argmin(np.abs(height - center_level))

x_center_ref = centers_x[center_idx]
y_center_ref = centers_y[center_idx]


# ======================================================
# COMPUTE TILT RELATIVE TO 2 KM CENTER
# ======================================================

x_rel = centers_x - x_center_ref
y_rel = centers_y - y_center_ref

tilt_mag = np.sqrt(x_rel**2 + y_rel**2)


# ======================================================
# FIND HIGHEST VALID CENTROID
# ======================================================

valid_idx = np.where(~np.isnan(centers_x))[0]
top_idx = valid_idx[-1]


# ======================================================
# SHIFT PLAN VIEW GRID
# ======================================================

east_shifted  = east  - x_center_ref
north_shifted = north - y_center_ref


# ======================================================
# CREATE FIGURE
# ======================================================

fig, axes = plt.subplots(1,2,figsize=(14,6))

fig.suptitle(
    f"Vortex Tilt Analysis | Case {case} | {utc_string}",
    fontsize=16,
    fontweight="bold"
)


# ======================================================
# LEFT PANEL: TILT PROFILE
# ======================================================

axes[0].plot(tilt_mag, height, linewidth=2)

axes[0].scatter(
    tilt_mag[center_idx],
    height[center_idx],
    color="red",
    label="2 km Reference"
)

axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile (Relative to 2 km Center)")
axes[0].grid(True)
axes[0].legend()


# ======================================================
# RIGHT PANEL: PLAN VIEW
# ======================================================

height_idx = np.argmin(np.abs(height - 1.0))

im = axes[1].imshow(
    refl_3d[:,:,height_idx],
    origin="lower",
    extent=[east_shifted[0], east_shifted[-1],
            north_shifted[0], north_shifted[-1]],
    cmap="turbo",
    vmin=-10,
    vmax=50
)


# Plot centroid positions
levels_to_plot = [0,3,6,9,12]
idx_levels = [np.argmin(np.abs(height-h)) for h in levels_to_plot]

for i in idx_levels:
    if not np.isnan(x_rel[i]):
        axes[1].scatter(
            x_rel[i],
            y_rel[i],
            s=120,
            label=f"{height[i]:.1f} km"
        )


# Plot bulk tilt vector
axes[1].plot(
    [0, x_rel[top_idx]],
    [0, y_rel[top_idx]],
    "k--",
    linewidth=2,
    label="Bulk Tilt Vector"
)


# Mark 2-km vortex center
axes[1].scatter(
    0,0,
    color="red",
    s=150,
    label="2 km Center"
)


axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_xlabel("Eastward Distance (km)")
axes[1].set_ylabel("Northward Distance (km)")
axes[1].set_title("Vortex Tilt Structure (Plan View)")
axes[1].legend()

fig.colorbar(im, ax=axes[1], label="Reflectivity (dBZ)")


# ======================================================
# FINAL LAYOUT
# ======================================================

plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 357
zoom_radius = 100   # km


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


# ======================================================
# LOAD RADAR DATA
# ======================================================
refl_3d = ds_swath['recentered_reflectivity'].values[case]

east  = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values


# ======================================================
# GET SWATH TIMES (RADAR DATA)  ✅ FIXED
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

start_time = pd.Timestamp(
    int(ds_swath['beg_swath_year'].values[case]),
    int(ds_swath['beg_swath_month'].values[case]),
    int(ds_swath['beg_swath_day'].values[case]),
    int(ds_swath['beg_swath_hour'].values[case]),
    int(ds_swath['beg_swath_min'].values[case])
)

end_time = pd.Timestamp(
    int(ds_swath['end_swath_year'].values[case]),
    int(ds_swath['end_swath_month'].values[case]),
    int(ds_swath['end_swath_day'].values[case]),
    int(ds_swath['end_swath_hour'].values[case]),
    int(ds_swath['end_swath_min'].values[case])
)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")

print("------ Swath Timing ------")
print("Start :", start_time)
print("Center:", center_time)
print("End   :", end_time)


# ======================================================
# CREATE GRID
# ======================================================
X, Y = np.meshgrid(east, north)


# ======================================================
# COMPUTE CENTROIDS BY HEIGHT
# ======================================================
centers_x = []
centers_y = []

for k in range(len(height)):

    refl_slice = refl_3d[:,:,k]

    mask = refl_slice >= 20

    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    weights = refl_slice * mask

    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ======================================================
# DEFINE 2-KM VORTEX CENTER
# ======================================================
center_level = 2.0
center_idx = np.argmin(np.abs(height - center_level))

x_center_ref = centers_x[center_idx]
y_center_ref = centers_y[center_idx]


# ======================================================
# COMPUTE TILT RELATIVE TO 2 KM CENTER
# ======================================================
x_rel = centers_x - x_center_ref
y_rel = centers_y - y_center_ref

tilt_mag = np.sqrt(x_rel**2 + y_rel**2)


# ======================================================
# FIND HIGHEST VALID CENTROID
# ======================================================
valid_idx = np.where(~np.isnan(centers_x))[0]
top_idx = valid_idx[-1]


# ======================================================
# SHIFT PLAN VIEW GRID
# ======================================================
east_shifted  = east  - x_center_ref
north_shifted = north - y_center_ref


# ======================================================
# CREATE FIGURE
# ======================================================
fig, axes = plt.subplots(1,2,figsize=(14,6))

fig.suptitle(
    f"Vortex Tilt Analysis | Case {case} | {utc_string}",
    fontsize=16,
    fontweight="bold"
)


# ======================================================
# LEFT PANEL: TILT PROFILE
# ======================================================
axes[0].plot(tilt_mag, height, linewidth=2)

axes[0].scatter(
    tilt_mag[center_idx],
    height[center_idx],
    color="red",
    label="2 km Reference"
)

axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile (Relative to 2 km Center)")
axes[0].grid(True)
axes[0].legend()


# ======================================================
# RIGHT PANEL: PLAN VIEW
# ======================================================
height_idx = np.argmin(np.abs(height - 1.0))

im = axes[1].imshow(
    refl_3d[:,:,height_idx],
    origin="lower",
    extent=[east_shifted[0], east_shifted[-1],
            north_shifted[0], north_shifted[-1]],
    cmap="turbo",
    vmin=-10,
    vmax=50
)


# Plot centroid positions
levels_to_plot = [0,3,6,9,12]
idx_levels = [np.argmin(np.abs(height-h)) for h in levels_to_plot]

for i in idx_levels:
    if not np.isnan(x_rel[i]):
        axes[1].scatter(
            x_rel[i],
            y_rel[i],
            s=120,
            label=f"{height[i]:.1f} km"
        )


# Plot bulk tilt vector
axes[1].plot(
    [0, x_rel[top_idx]],
    [0, y_rel[top_idx]],
    "k--",
    linewidth=2,
    label="Bulk Tilt Vector"
)


# Mark 2-km vortex center
axes[1].scatter(
    0,0,
    color="red",
    s=150,
    label="2 km Center"
)


axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_xlabel("Eastward Distance (km)")
axes[1].set_ylabel("Northward Distance (km)")
axes[1].set_title("Vortex Tilt Structure (Plan View)")
axes[1].legend()

fig.colorbar(im, ax=axes[1], label="Reflectivity (dBZ)")


# ======================================================
# FINAL LAYOUT
# ======================================================
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

In [ ]:
# CALCULATE CIRCULATION

In [ ]:
level_for_circ = 1.0  # km (you can change this)
level_idx = np.argmin(np.abs(height - level_for_circ))

zeta = ds_swath['recentered_relative_vorticity'].values[case, :, :, level_idx]

max_idx = np.nanargmax(zeta)

iy, ix = np.unravel_index(max_idx, zeta.shape)

x_circ = east[ix]
y_circ = north[iy]

level_for_circ = 1.0

In [ ]:
from scipy.ndimage import gaussian_filter

zeta_smooth = gaussian_filter(zeta, sigma=2)

max_idx = np.nanargmax(zeta_smooth)

In [ ]:
print(f"Using height = {height[level_idx]:.2f} km")
print(f"Circulation center: x={x_circ:.2f}, y={y_circ:.2f}")

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 351
zoom_radius = 100   # km


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter


# ======================================================
# LOAD RADAR DATA
# ======================================================
refl_3d = ds_swath['recentered_reflectivity'].values[case]

east  = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values


# ======================================================
# GET SWATH TIMES
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")


# ======================================================
# CREATE GRID
# ======================================================
X, Y = np.meshgrid(east, north)


# ======================================================
# COMPUTE CENTROIDS BY HEIGHT
# ======================================================
centers_x = []
centers_y = []

for k in range(len(height)):

    refl_slice = refl_3d[:,:,k]
    mask = refl_slice >= 20

    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    weights = refl_slice * mask

    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ======================================================
# DEFINE 2-KM VORTEX CENTER
# ======================================================
center_level = 2.0
center_idx = np.argmin(np.abs(height - center_level))

x_center_ref = centers_x[center_idx]
y_center_ref = centers_y[center_idx]


# ======================================================
# COMPUTE TILT
# ======================================================
x_rel = centers_x - x_center_ref
y_rel = centers_y - y_center_ref
tilt_mag = np.sqrt(x_rel**2 + y_rel**2)

valid_idx = np.where(~np.isnan(centers_x))[0]
top_idx = valid_idx[-1]


# ======================================================
# SHIFT GRID
# ======================================================
east_shifted  = east  - x_center_ref
north_shifted = north - y_center_ref


# ======================================================
# COMPUTE VORTICITY CENTER (1 km)
# ======================================================
level_idx = np.argmin(np.abs(height - 1.0))

zeta = ds_swath['recentered_relative_vorticity'].values[case, :, :, level_idx]

# Smooth to remove noise
zeta_smooth = gaussian_filter(zeta, sigma=2)

# Find max vorticity location
max_idx = np.nanargmax(zeta_smooth)
iy, ix = np.unravel_index(max_idx, zeta.shape)

x_circ = east[ix]
y_circ = north[iy]

print(f"Using height = {height[level_idx]:.2f} km")
print(f"Vorticity center: x={x_circ:.2f}, y={y_circ:.2f}")


# ======================================================
# CREATE FIGURE
# ======================================================
fig, axes = plt.subplots(1,2,figsize=(14,6))

fig.suptitle(
    f"Vortex Tilt + Circulation | Case {case} | {utc_string}",
    fontsize=16,
    fontweight="bold"
)


# ======================================================
# LEFT: TILT PROFILE
# ======================================================
axes[0].plot(tilt_mag, height, linewidth=2)

axes[0].scatter(
    tilt_mag[center_idx],
    height[center_idx],
    color="red",
    label="2 km Reference"
)

axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile")
axes[0].grid(True)
axes[0].legend()


# ======================================================
# RIGHT: PLAN VIEW
# ======================================================
height_idx = level_idx  # keep consistent (1 km)

im = axes[1].imshow(
    refl_3d[:,:,height_idx],
    origin="lower",
    extent=[east_shifted[0], east_shifted[-1],
            north_shifted[0], north_shifted[-1]],
    cmap="turbo",
    vmin=-10,
    vmax=50
)


# --- VORTICITY CONTOURS ---
axes[1].contour(
    east_shifted,
    north_shifted,
    zeta_smooth,
    levels=10,
    linewidths=1
)


# --- CENTROIDS ---
levels_to_plot = [0,3,6,9,12]
idx_levels = [np.argmin(np.abs(height-h)) for h in levels_to_plot]

for i in idx_levels:
    if not np.isnan(x_rel[i]):
        axes[1].scatter(
            x_rel[i],
            y_rel[i],
            s=120,
            label=f"{height[i]:.1f} km"
        )


# --- BULK TILT VECTOR ---
axes[1].plot(
    [0, x_rel[top_idx]],
    [0, y_rel[top_idx]],
    "k--",
    linewidth=2,
    label="Bulk Tilt"
)


# --- 2 KM CENTER ---
axes[1].scatter(
    0, 0,
    color="red",
    s=150,
    label="2 km Center"
)


# --- VORTICITY CENTER ---
axes[1].scatter(
    x_circ - x_center_ref,
    y_circ - y_center_ref,
    s=220,
    marker="X",
    label="Max Vorticity (1 km)"
)


# ======================================================
# AXES FORMATTING
# ======================================================
axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_xlabel("Eastward Distance (km)")
axes[1].set_ylabel("Northward Distance (km)")
axes[1].set_title("Reflectivity + Vorticity Structure")
axes[1].legend()

fig.colorbar(im, ax=axes[1], label="Reflectivity (dBZ)")


# ======================================================
# FINAL
# ======================================================
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 353
zoom_radius = 100   # km


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter


# ======================================================
# LOAD RADAR DATA
# ======================================================
refl_3d = ds_swath['recentered_reflectivity'].values[case]

east  = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values


# ======================================================
# SWATH TIME
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")


# ======================================================
# GRID
# ======================================================
X, Y = np.meshgrid(east, north)


# ======================================================
# CENTROIDS BY HEIGHT
# ======================================================
centers_x, centers_y = [], []

for k in range(len(height)):

    refl_slice = refl_3d[:, :, k]
    mask = refl_slice >= 20

    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    weights = refl_slice * mask

    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ======================================================
# 2-KM REFERENCE CENTER
# ======================================================
center_level = 2.0
center_idx = np.argmin(np.abs(height - center_level))

x_center_ref = centers_x[center_idx]
y_center_ref = centers_y[center_idx]


# ======================================================
# TILT
# ======================================================
x_rel = centers_x - x_center_ref
y_rel = centers_y - y_center_ref
tilt_mag = np.sqrt(x_rel**2 + y_rel**2)

valid_idx = np.where(~np.isnan(centers_x))[0]
top_idx = valid_idx[-1]


# ======================================================
# SHIFT GRID
# ======================================================
east_shifted  = east - x_center_ref
north_shifted = north - y_center_ref

X2, Y2 = np.meshgrid(east_shifted, north_shifted)


# ======================================================
# VORTICITY (1 km)
# ======================================================
level_idx = np.argmin(np.abs(height - 1.0))

zeta = ds_swath['recentered_relative_vorticity'].values[case, :, :, level_idx]
zeta_smooth = gaussian_filter(zeta, sigma=2)


# ======================================================
# MAX VORTICITY LOCATION
# ======================================================
max_idx = np.nanargmax(zeta_smooth)
iy, ix = np.unravel_index(max_idx, zeta_smooth.shape)

x_circ = east_shifted[ix]
y_circ = north_shifted[iy]
max_zeta = zeta_smooth[iy, ix]


# ======================================================
# VORTICITY GRADIENT FIELD
# ======================================================
dx = np.mean(np.diff(east_shifted))
dy = np.mean(np.diff(north_shifted))

dz_dy, dz_dx = np.gradient(zeta_smooth, dy, dx)


# subsample for clean quiver
step = 6

Xq = X2[::step, ::step]
Yq = Y2[::step, ::step]
U = dz_dx[::step, ::step]
V = dz_dy[::step, ::step]


# ======================================================
# FIGURE
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fig.suptitle(
    f"Vortex Tilt + Vorticity Structure | Case {case} | {utc_string}",
    fontsize=16,
    fontweight="bold"
)


# ======================================================
# LEFT: TILT PROFILE
# ======================================================
axes[0].plot(tilt_mag, height, linewidth=2)

axes[0].scatter(
    tilt_mag[center_idx],
    height[center_idx],
    color="red",
    label="2 km Reference"
)

axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile")
axes[0].grid(True)
axes[0].legend()


# ======================================================
# RIGHT: PLAN VIEW
# ======================================================
im = axes[1].imshow(
    refl_3d[:, :, level_idx],
    origin="lower",
    extent=[east_shifted[0], east_shifted[-1],
            north_shifted[0], north_shifted[-1]],
    cmap="turbo",
    vmin=-10,
    vmax=50
)


# ======================================================
# CENTROIDS
# ======================================================
levels_to_plot = [0, 3, 6, 9]
idx_levels = [np.argmin(np.abs(height - h)) for h in levels_to_plot]

for i in idx_levels:
    if not np.isnan(x_rel[i]):
        axes[1].scatter(
            x_rel[i],
            y_rel[i],
            s=120,
            label=f"{height[i]:.1f} km"
        )


# ======================================================
# BULK TILT
# ======================================================
axes[1].plot(
    [0, x_rel[top_idx]],
    [0, y_rel[top_idx]],
    "k--",
    linewidth=2,
    label="Bulk Tilt"
)


# ======================================================
# 2-KM CENTER
# ======================================================
axes[1].scatter(
    0, 0,
    color="red",
    s=150,
    label="2 km Center"
)


# ======================================================
# MAX VORTICITY (X)
# ======================================================
axes[1].scatter(
    x_circ,
    y_circ,
    s=300,
    marker="X",
    color="black",
    label="Max Vorticity (1 km)"
)


# ======================================================
# ANNOTATION (BOTTOM CORNER)
# ======================================================
axes[1].text(
    0.02, 0.02,
    f"Max ζ = {max_zeta:.2e} s⁻¹",
    transform=axes[1].transAxes,
    fontsize=10,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
)


# ======================================================
# FORMATTING
# ======================================================
axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_xlabel("Eastward Distance (km)")
axes[1].set_ylabel("Northward Distance (km)")
axes[1].set_title("Reflectivity + Vorticity Dynamics")

axes[1].legend()
fig.colorbar(im, ax=axes[1], label="Reflectivity (dBZ)")


# ======================================================
# FINAL
# ======================================================
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 350
zoom_radius = 100   # km


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter


# ======================================================
# LOAD RADAR DATA
# ======================================================
refl_3d = ds_swath['recentered_reflectivity'].values[case]

east  = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values


# ======================================================
# SWATH TIME
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")


# ======================================================
# GRID
# ======================================================
X, Y = np.meshgrid(east, north)


# ======================================================
# CENTROIDS BY HEIGHT
# ======================================================
centers_x, centers_y = [], []

for k in range(len(height)):

    refl_slice = refl_3d[:, :, k]
    mask = refl_slice >= 20

    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    weights = refl_slice * mask

    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ======================================================
# 2-KM REFERENCE CENTER
# ======================================================
center_level = 2.0
center_idx = np.argmin(np.abs(height - center_level))

x_center_ref = centers_x[center_idx]
y_center_ref = centers_y[center_idx]


# ======================================================
# TILT
# ======================================================
x_rel = centers_x - x_center_ref
y_rel = centers_y - y_center_ref
tilt_mag = np.sqrt(x_rel**2 + y_rel**2)

valid_idx = np.where(~np.isnan(centers_x))[0]
top_idx = valid_idx[-1]


# ======================================================
# SHIFT GRID
# ======================================================
east_shifted  = east - x_center_ref
north_shifted = north - y_center_ref

X2, Y2 = np.meshgrid(east_shifted, north_shifted)


# ======================================================
# VORTICITY (1 km)
# ======================================================
level_idx = np.argmin(np.abs(height - 1.0))

zeta = ds_swath['recentered_relative_vorticity'].values[case, :, :, level_idx]
zeta_smooth = gaussian_filter(zeta, sigma=2)


# ======================================================
# FIGURE
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fig.suptitle(
    f"Vortex Tilt + Reflectivity | Case {case} | {utc_string}",
    fontsize=16,
    fontweight="bold"
)


# ======================================================
# LEFT: TILT PROFILE
# ======================================================
axes[0].plot(tilt_mag, height, linewidth=2)

axes[0].scatter(
    tilt_mag[center_idx],
    height[center_idx],
    color="red",
    label="2 km Reference"
)

axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile")
axes[0].grid(True)
axes[0].legend()


# ======================================================
# RIGHT: PLAN VIEW (CLEAN REFLECTIVITY ONLY)
# ======================================================
im = axes[1].imshow(
    refl_3d[:, :, level_idx],
    origin="lower",
    extent=[east_shifted[0], east_shifted[-1],
            north_shifted[0], north_shifted[-1]],
    cmap="turbo",
    vmin=-10,
    vmax=50
)

axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_title("Reflectivity (dBZ)")

# Optional: remove ticks/labels for cleaner look
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].set_xlabel("")
axes[1].set_ylabel("")

fig.colorbar(im, ax=axes[1], label="Reflectivity (dBZ)")


# ======================================================
# FINAL
# ======================================================
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# ======================================================
# USER INPUT
# ======================================================
case = 350
zoom_radius = 100   # km
max_height_km = 8.0


# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter


# ======================================================
# LOAD RADAR DATA
# ======================================================
refl_3d_full = ds_swath['recentered_reflectivity'].values[case]

east  = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height_full = ds_swath['height'].values


# ======================================================
# LIMIT HEIGHT TO 8 KM
# ======================================================
height_mask = height_full <= max_height_km

height = height_full[height_mask]
refl_3d = refl_3d_full[:, :, height_mask]


# ======================================================
# SWATH TIME
# ======================================================
center_time = pd.Timestamp(
    int(ds_swath['swath_year'].values[case]),
    int(ds_swath['swath_month'].values[case]),
    int(ds_swath['swath_day'].values[case]),
    int(ds_swath['swath_hour'].values[case]),
    int(ds_swath['swath_min'].values[case])
)

utc_string = center_time.strftime("%Y-%m-%d %H:%M UTC")


# ======================================================
# GRID
# ======================================================
X, Y = np.meshgrid(east, north)


# ======================================================
# CENTROIDS BY HEIGHT (NOW ONLY ≤ 8 KM)
# ======================================================
centers_x, centers_y = [], []

for k in range(len(height)):

    refl_slice = refl_3d[:, :, k]
    mask = refl_slice >= 20

    if np.sum(mask) < 50:
        centers_x.append(np.nan)
        centers_y.append(np.nan)
        continue

    weights = refl_slice * mask

    x_center = np.nansum(X * weights) / np.nansum(weights)
    y_center = np.nansum(Y * weights) / np.nansum(weights)

    centers_x.append(x_center)
    centers_y.append(y_center)

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)


# ======================================================
# 2-KM REFERENCE CENTER
# ======================================================
center_level = 2.0
center_idx = np.argmin(np.abs(height - center_level))

x_center_ref = centers_x[center_idx]
y_center_ref = centers_y[center_idx]


# ======================================================
# TILT (ONLY UP TO 8 KM)
# ======================================================
x_rel = centers_x - x_center_ref
y_rel = centers_y - y_center_ref
tilt_mag = np.sqrt(x_rel**2 + y_rel**2)


# ======================================================
# SHIFT GRID
# ======================================================
east_shifted  = east - x_center_ref
north_shifted = north - y_center_ref

X2, Y2 = np.meshgrid(east_shifted, north_shifted)


# ======================================================
# VORTICITY (1 km, still valid)
# ======================================================
level_idx = np.argmin(np.abs(height - 1.0))

zeta = ds_swath['recentered_relative_vorticity'].values[case, :, :, level_idx]
zeta_smooth = gaussian_filter(zeta, sigma=2)


# ======================================================
# FIGURE
# ======================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fig.suptitle(
    f"Vortex Tilt (≤8 km) + Reflectivity | Case {case} | {utc_string}",
    fontsize=16,
    fontweight="bold"
)


# ======================================================
# LEFT: TILT PROFILE (LIMITED TO 8 KM)
# ======================================================
axes[0].plot(tilt_mag, height, linewidth=2)

axes[0].scatter(
    tilt_mag[center_idx],
    height[center_idx],
    color="red",
    label="2 km Reference"
)

axes[0].set_xlabel("Horizontal Displacement (km)")
axes[0].set_ylabel("Height (km)")
axes[0].set_title("Vortex Tilt Profile (≤ 8 km)")
axes[0].set_ylim(0, max_height_km)
axes[0].grid(True)
axes[0].legend()


# ======================================================
# RIGHT: PLAN VIEW (CLEAN REFLECTIVITY ONLY)
# ======================================================
im = axes[1].imshow(
    refl_3d[:, :, level_idx],
    origin="lower",
    extent=[east_shifted[0], east_shifted[-1],
            north_shifted[0], north_shifted[-1]],
    cmap="turbo",
    vmin=-10,
    vmax=50
)

axes[1].set_xlim(-zoom_radius, zoom_radius)
axes[1].set_ylim(-zoom_radius, zoom_radius)

axes[1].set_title("Reflectivity (dBZ)")

# clean look
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].set_xlabel("")
axes[1].set_ylabel("")

fig.colorbar(im, ax=axes[1], label="Reflectivity (dBZ)")


# ======================================================
# FINAL
# ======================================================
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter

# ======================================================
# USER INPUT
# ======================================================
max_height_km = 8.0
reference_height = 2.0  # km (reference level for tilt calculation)
min_reflectivity = 20   # dBZ threshold for centroid calculation


# ======================================================
# GRID SETUP
# ======================================================
east  = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height_full = ds_swath['height'].values

# Limit height to 8 km
height_mask = height_full <= max_height_km
height = height_full[height_mask]


# ======================================================
# EXTRACT SWATH TIMES AND CONVERT TO DATETIME
# ======================================================
n_cases = ds_swath['swath_year'].shape[0]
swath_times = []

for case in range(n_cases):
    try:
        time = pd.Timestamp(
            int(ds_swath['swath_year'].values[case]),
            int(ds_swath['swath_month'].values[case]),
            int(ds_swath['swath_day'].values[case]),
            int(ds_swath['swath_hour'].values[case]),
            int(ds_swath['swath_min'].values[case])
        )
        swath_times.append(time)
    except:
        swath_times.append(pd.NaT)

# Filter to study period (Sep 11 - Sep 17, 2014)
study_start = pd.Timestamp("2014-09-11 00:00:00")
study_end = pd.Timestamp("2014-09-17 23:59:00")

case_indices = []
case_times = []

for case, t in enumerate(swath_times):
    if pd.notna(t) and study_start <= t <= study_end:
        case_indices.append(case)
        case_times.append(t)

print(f"Found {len(case_indices)} radar passes in study period")
print(f"Cases: {case_indices}")
print(f"Times: {case_times}")


# ======================================================
# COMPUTE VORTEX TILT FOR EACH CASE
# ======================================================
tilt_magnitudes = []
tilt_directions = []
center_idx_ref = np.argmin(np.abs(height - reference_height))

for case in case_indices:
    
    # Extract 3D reflectivity for this case, limit to 8 km
    refl_3d_full = ds_swath['recentered_reflectivity'].values[case]
    refl_3d = refl_3d_full[:, :, height_mask]
    
    # Create grid
    X, Y = np.meshgrid(east, north)
    
    # Compute centroids at each height level
    centers_x, centers_y = [], []
    
    for k in range(len(height)):
        refl_slice = refl_3d[:, :, k]
        mask = refl_slice >= min_reflectivity
        
        if np.sum(mask) < 50:
            centers_x.append(np.nan)
            centers_y.append(np.nan)
            continue
        
        weights = refl_slice * mask
        x_center = np.nansum(X * weights) / np.nansum(weights)
        y_center = np.nansum(Y * weights) / np.nansum(weights)
        
        centers_x.append(x_center)
        centers_y.append(y_center)
    
    centers_x = np.array(centers_x)
    centers_y = np.array(centers_y)
    
    # Get reference center at 2 km
    x_ref = centers_x[center_idx_ref]
    y_ref = centers_y[center_idx_ref]
    
    # Compute tilt relative to 2 km level
    x_rel = centers_x - x_ref
    y_rel = centers_y - y_ref
    tilt_mag = np.sqrt(x_rel**2 + y_rel**2)
    
    # Compute RMS tilt magnitude from 2 km to 8 km (excluding NaN values)
    valid_tilt = tilt_mag[~np.isnan(tilt_mag)]
    if len(valid_tilt) > 0:
        rms_tilt = np.sqrt(np.mean(valid_tilt**2))
    else:
        rms_tilt = np.nan
    
    tilt_magnitudes.append(rms_tilt)
    
    # Compute mean direction (angle) of tilt
    valid_mask = ~(np.isnan(x_rel) | np.isnan(y_rel))
    if np.sum(valid_mask) > 0:
        mean_angle = np.arctan2(np.mean(y_rel[valid_mask]), np.mean(x_rel[valid_mask]))
        tilt_directions.append(np.degrees(mean_angle))
    else:
        tilt_directions.append(np.nan)


# ======================================================
# CREATE DATAFRAME FOR TIME SERIES
# ======================================================
df_tilt = pd.DataFrame({
    'datetime': case_times,
    'tilt_km': tilt_magnitudes,
    'tilt_direction_deg': tilt_directions
})

df_tilt = df_tilt.sort_values('datetime').reset_index(drop=True)

print("\nVortex Tilt Time Series:")
print(df_tilt)


# ======================================================
# PLOT TIME SERIES
# ======================================================
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)

ax.plot(df_tilt['datetime'], df_tilt['tilt_km'], 
        marker='o', markersize=8, linewidth=2.5, 
        color='#185FA5', label='RMS Vortex Tilt')

ax.axhline(y=40, color='green', linestyle='--', linewidth=1.5, 
           label='Small tilt threshold (40 km)', alpha=0.7)
ax.axhline(y=100, color='red', linestyle='--', linewidth=1.5, 
           label='Large tilt threshold (100 km)', alpha=0.7)

ax.fill_between(df_tilt['datetime'], 0, 40, color='green', alpha=0.1, label='Favorable')
ax.fill_between(df_tilt['datetime'], 40, 100, color='yellow', alpha=0.1)
ax.fill_between(df_tilt['datetime'], 100, ax.get_ylim()[1], color='red', alpha=0.1, label='Unfavorable')

ax.set_xlabel('Time (UTC)', fontsize=12, fontweight='bold')
ax.set_ylabel('RMS Vortex Tilt (km)', fontsize=12, fontweight='bold')
ax.set_title('Hurricane Edouard: Vortex Tilt Evolution (Sep 11-17, 2014)\n(0-8 km altitude)', 
             fontsize=14, fontweight='bold')

ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()

print("\n✅ Time series complete")